In [1]:
import numpy as np
import xarray as xr

# Processing Larsim

## Disaggregate the 3h accumulated variables

In [2]:
def diff_within_blocks_vectorized(da, block_size=3):
    """
    Disaggregates the 3h accumulated variables produced by the assimilation system.
    The logic is: [(0),1,2,3], [(3),4,5,6], [(6),7,8,9], ... UTC are blocks. Within each block,
    some variables are accumulated, e.g., total precipitaiton, initialized from zero (values in brackets).
    For example, the hourly precipitation would be 1 UTC - 0 UTC (all zero), 2 UTC - 1 UTC, 
    3 UTC - 2 UTC. Note that this creates an overlap, but we are only processing/loading data on the meaningful
    timesteps.
    """

    # Create groups based on 3h intervals:
    block_labels = np.zeros(len(da["time"]), dtype=int)

    i_block = 0
    for i, dt in enumerate(da.time):
        if dt.dt.hour in [1, 4, 7, 10, 13, 16, 19, 22]:
            i_block += 1
        
        block_labels[i] = i_block
    
    da_with_blocks = da.assign_coords(block=('time', block_labels)) #add block labels as a coordinate
 
    def block_diff(block):
        if len(block.time) > 1:
            shifted = block.shift(time=1).values
            shifted[0] = 0.
            
            return block - shifted
        return block
    
    # Apply the function to each block and concatenate results
    result = da_with_blocks.groupby('block').map(block_diff)
    
    return result.drop_vars('block') #drop variables

In [ ]:
path_data = "/automount/agh/s6tifohr/july21_eval/data/LARSIM/"
exp_name = "blcklst_sat"
ds = xr.open_mfdataset([f"{path_data}/{exp_name}/daily_files/fc_R03B08_N02_sel.202107{dd:02}.nc" for dd in range(6,17)])

In [ ]:
# Disaggregate grid scale rain
da_diff_rain_gsp = diff_within_blocks_vectorized(ds["RAIN_GSP"])
ds["RAIN_GSP"] = da_diff_rain_gsp

# Disaggregate total precipitation
da_diff_tot_prec = diff_within_blocks_vectorized(ds["TOT_PREC"])
ds["TOT_PREC"] = da_diff_tot_prec

In [ ]:
# Output daily files:
for dd in range(6,17):
    time_slice = slice(np.datetime64(f"2021-07-{dd:02}T00"), np.datetime64(f"2021-07-{dd:02}T23"))
    path_out = f"{path_data}/{exp_name}/disaggregated/larsim_{exp_name}_det_202107{dd:02}.nc"
    ds.sel(time=time_slice).to_netcdf(path_out)